<a href="https://colab.research.google.com/github/letiBri/MaskArchitectureAnomaly_CourseProject/blob/main/STEP8_CocoFineTuned.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Anomaly Segmentation Eomt Coco Fine-tuned (Step 8)

In [1]:
!git clone https://github.com/letiBri/MaskArchitectureAnomaly_CourseProject.git
%cd MaskArchitectureAnomaly_CourseProject/eomt

fatal: destination path 'MaskArchitectureAnomaly_CourseProject' already exists and is not an empty directory.
/content/MaskArchitectureAnomaly_CourseProject/eomt


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import sys
sys.path.append('/content/MaskArchitectureAnomaly_CourseProject')
sys.path.append('/content/MaskArchitectureAnomaly_CourseProject/eval')
sys.path.append('/content/MaskArchitectureAnomaly_CourseProject/eomt')


In [4]:
!pip uninstall -y torch torchvision torchaudio

# 2. Installiamo versioni di PyTorch perfettamente accoppiate (CUDA 12.4, standard su Colab)
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

# 3. Installiamo le metriche e lightning
!pip install ood-metrics lightning

# 4. Rimuoviamo torch e torchvision dal requirements.txt per evitare che lo rovini di nuovo
!sed -i '/torch==/d' requirements.txt
!sed -i '/torchvision==/d' requirements.txt

# 5. Installiamo il resto dei requirements puliti
!python3 -m pip install -r requirements.txt

Found existing installation: torch 2.10.0+cpu
Uninstalling torch-2.10.0+cpu:
  Successfully uninstalled torch-2.10.0+cpu
Found existing installation: torchvision 0.25.0+cpu
Uninstalling torchvision-0.25.0+cpu:
  Successfully uninstalled torchvision-0.25.0+cpu
Found existing installation: torchaudio 2.10.0+cpu
Uninstalling torchaudio-2.10.0+cpu:
  Successfully uninstalled torchaudio-2.10.0+cpu
Looking in indexes: https://download.pytorch.org/whl/cu124
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 26.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 28.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 35.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 

  Installing build dependencies ... canceled
ERROR: Operation cancelled by user
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 377, in run
^C


In [4]:
import os
import cv2
import glob
import torch
import random
from PIL import Image
import numpy as np
import os.path as osp
from argparse import ArgumentParser
from ood_metrics import fpr_at_95_tpr, calc_metrics, plot_roc, plot_pr,plot_barcode
from sklearn.metrics import roc_auc_score, roc_curve, auc, precision_recall_curve, average_precision_score
from torchvision.transforms import Compose, Resize, ToTensor, Normalize

# Setup

In [6]:
import yaml
from lightning import seed_everything
import torch
from torch.nn import functional as F
from torch.amp.autocast_mode import autocast
import matplotlib.pyplot as plt
import numpy as np
from huggingface_hub import hf_hub_download
from huggingface_hub.utils import RepositoryNotFoundError
import warnings
import importlib

seed_everything(0, verbose=False)

device = 0  # change to the GPU you want to use
IGNORE_INDEX = 255
img_idx = 10  # change to the index of the image you want to visualize
data_path = "/content/drive/MyDrive/CourseProjectAnomaly"  # drive folder of the cityscapes val set per trovare la mIoU

with open("configs/dinov2/cityscapes/semantic/eomt_base_640.yaml", "r") as f:
    config_cs = yaml.safe_load(f)


# Load Dataset


In [7]:
data_module_name, class_name = config_cs["data"]["class_path"].rsplit(".", 1)
data_module = getattr(importlib.import_module(data_module_name), class_name)
data_module_kwargs = config_cs["data"].get("init_args", {})

data = data_module(
    path=data_path,
    batch_size=1,
    num_workers=0,
    check_empty_targets=False,
    **data_module_kwargs
)
data.setup()

# Load model

In [8]:
current_config = config_cs

####
# 1. Inserisci il percorso del file generato dal tuo fine-tuning (probabilmente un .ckpt)
percorso_checkpoint = "/content/drive/MyDrive/CourseProjectAnomaly/checkpoints/last.ckpt"

# 2. Scegli dove salvare il nuovo file .bin pulito
percorso_bin_output = "/content/drive/MyDrive/CourseProjectAnomaly/eomt_coco_finetuned.bin"

# Lo carichiamo sulla CPU per evitare problemi di memoria video
checkpoint = torch.load(percorso_checkpoint, map_location="cpu", weights_only=False)

# PyTorch Lightning di solito salva i pesi dentro una chiave chiamata "state_dict"
if "state_dict" in checkpoint:
    state_dict_pulito = checkpoint["state_dict"]
else:
    # Nel caso in cui il file fosse già solo un dizionario di pesi
    state_dict_pulito = checkpoint

# Salviamo solo il dizionario dei pesi
torch.save(state_dict_pulito, percorso_bin_output)
####



target_img_size = (640, 640)
num_classes_to_load = 19
state_dict_path = percorso_bin_output

###
num_queries_to_load = 200 # forzato per coco, dato che la config di cityscapes se ne aspetta 100
###

warnings.filterwarnings(
    "ignore",
    message=r".*Attribute 'network' is an instance of `nn\.Module` and is already saved during checkpointing.*",
)

# Load encoder
encoder_cfg = current_config["model"]["init_args"]["network"]["init_args"]["encoder"]
encoder_module_name, encoder_class_name = encoder_cfg["class_path"].rsplit(".", 1)
encoder_cls = getattr(importlib.import_module(encoder_module_name), encoder_class_name)
encoder = encoder_cls(img_size=target_img_size, **encoder_cfg.get("init_args", {}))

# Load network
network_cfg = current_config["model"]["init_args"]["network"]
network_module_name, network_class_name = network_cfg["class_path"].rsplit(".", 1)
network_cls = getattr(importlib.import_module(network_module_name), network_class_name)


### Estraiamo i kwargs della rete, ma SOVRASCRIVIAMO le query!
network_kwargs = {k: v for k, v in network_cfg["init_args"].items() if k != "encoder"}
network_kwargs["num_q"] = num_queries_to_load # <-- Sostituisce il 100 con il 200!
network = network_cls(
    masked_attn_enabled=False,
    num_classes=num_classes_to_load,
    encoder=encoder,
    **network_kwargs,
)
###


# Load Lightning module
lit_module_name, lit_class_name = current_config["model"]["class_path"].rsplit(".", 1)
lit_cls = getattr(importlib.import_module(lit_module_name), lit_class_name)
model_kwargs = {k: v for k, v in current_config["model"]["init_args"].items() if k != "network"}

if "stuff_classes" in current_config["data"].get("init_args", {}):
    model_kwargs["stuff_classes"] = current_config["data"]["init_args"]["stuff_classes"]

model = (
    lit_cls(
        img_size=target_img_size,
        num_classes=num_classes_to_load,
        network=network,
        **model_kwargs,
    )
    .eval()
    .to(device)
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

RuntimeError: Cannot access accelerator device when none is available.

# Load Pretrained

In [ ]:
model_kwargs_final = {k: v for k, v in model_kwargs.items()}

# FIX CRUCIALE: Rimuoviamo eventuali 'num_classes' dai kwargs che potrebbero sovrascrivere il nostro valore
if "num_classes" in model_kwargs_final:
  del model_kwargs_final["num_classes"]

name = current_config.get("trainer", {}).get("logger", {}).get("init_args", {}).get("name")
is_dinov3 = "dinov3" in name if name else False

if is_dinov3:
    model_kwargs["ckpt_path"] = state_dict_path
    model_kwargs["delta_weights"] = True

model = (
    lit_cls(
        img_size=target_img_size,
        num_classes=num_classes_to_load,
        network=network,
        **model_kwargs_final,
    )
    .eval()
    .to(device)
)

if not is_dinov3:
    state_dict = torch.load(
        state_dict_path, map_location=f"cuda:{device}", weights_only=True)
    model.load_state_dict(state_dict, strict=False)

RuntimeError: Error(s) in loading state_dict for MaskClassificationSemantic:
	size mismatch for network.q.weight: copying a param with shape torch.Size([200, 768]) from checkpoint, the shape in current model is torch.Size([100, 768]).

# Semantic Inference

In [ ]:
IGNORE_INDEX = 255

# DEFINIZIONE MAPPATURA UNIFICATA (COCO -> CITYSCAPES TRAIN_IDS)
# Questo dizionario mappa i category_id di COCO nei trainId di Cityscapes (0-18)
coco_to_cityscapes_map = {
    # --- THINGS (Istanze) ---
    0: 11,   # person -> person
    1: 18,   # bicycle -> bicycle
    2: 13,   # car -> car
    3: 17,   # motorcycle -> motorcycle
    5: 15,   # bus -> bus
    6: 16,   # train -> train
    7: 14,   # truck -> truck
    9: 6,    # traffic light -> traffic light
    11: 7,   # stop sign -> traffic sign (generico)

    # --- STUFF (Sfondi/Regioni) ---
    # Nota: Assicurati che questi ID corrispondano al file JSON di COCO Stuff fornito
    100: 0,  # road -> road
    123: 1,  # pavement-merged -> sidewalk
    129: 2,  # building-other-merged -> building
    91: 2,   # house -> building
    131: 3,  # wall-other-merged -> wall
    117: 4,  # fence-merged -> fence
    116: 8,  # tree-merged -> vegetation
    125: 8,  # grass-merged -> vegetation
    90: 9,   # gravel -> terrain
    102: 9,  # sand -> terrain
    119: 10, # sky-other-merged -> sky
}

def translate_preds(pred_array):
    """
    Converte i label predetti da COCO (0-132) nei label Cityscapes (0-18).
    Tutto ciò che non è nel mapping diventa IGNORE_INDEX (255).
    """
    translated = np.full_like(pred_array, fill_value=IGNORE_INDEX)
    for coco_id, city_id in coco_to_cityscapes_map.items():
        translated[pred_array == coco_id] = city_id
    return translated

def infer_semantic(img, target):
    model.eval()

    # Forza il modello a usare la dimensione della finestra corretta
    # (640 per COCO, 1024 per Cityscapes)
    model.window_size = target_img_size[0]

    with torch.no_grad(), autocast(dtype=torch.float16, device_type="cuda"):
        imgs = [img.to(device)]
        img_sizes = [img.shape[-2:] for img in imgs]
        crops, origins = model.window_imgs_semantic(imgs)

        mask_logits_per_layer, class_logits_per_layer = model(crops)
        mask_logits = F.interpolate(
            mask_logits_per_layer[-1], target_img_size, mode="bilinear"
        )

        crop_logits = model.to_per_pixel_logits_semantic(
            mask_logits, class_logits_per_layer[-1]
        )
        logits = model.revert_window_logits_semantic(crop_logits, origins, img_sizes)
        preds = logits[0].argmax(0).cpu()

    pred_array = preds.numpy()

    # Se il modello caricato è COCO (134 classi), traduciamo i risultati
    if use_coco: # se siamo nel modello coco
        pred_array = translate_preds(pred_array)

    target_array = model.to_per_pixel_targets_semantic([target], IGNORE_INDEX)[0].numpy()
    return pred_array, target_array

# Evaluation mIoU

In [ ]:
import numpy as np
from tqdm import tqdm

def fast_hist(a, b, n):
    k = (a >= 0) & (a < n) & (b >= 0) & (b < n)
    return np.bincount(n * a[k].astype(int) + b[k], minlength=n**2).reshape(n, n)

def per_class_iu(hist):
    return np.diag(hist) / (hist.sum(1) + hist.sum(0) - np.diag(hist))

# setting
num_eval_classes = 19
hist = np.zeros((num_eval_classes, num_eval_classes))
val_loader = data.val_dataloader()

print(f"evaluation on {len(val_loader)} images...")
print(f"Current model: {'COCO' if use_coco else 'Cityscapes'}")

# evaluation
for i, batch in enumerate(tqdm(val_loader)):
    img = batch[0][0]
    target = batch[1][0]

    pred_array, target_array = infer_semantic(img, target)

    hist += fast_hist(target_array.flatten(), pred_array.flatten(), num_eval_classes)

# calcolo finale
ious = per_class_iu(hist)
miou = np.nanmean(ious)

# visualization results
classes = [
    "road", "sidewalk", "building", "wall", "fence", "pole", "traffic light",
    "traffic sign", "vegetation", "terrain", "sky", "person", "rider", "car",
    "truck", "bus", "train", "motorcycle", "bicycle"
]

print("\n" + "="*30)
print(f"Results MIOU - {'COCO' if use_coco else 'CITYSCAPES'}")
print("="*30)
for name, iou in zip(classes, ious):
    print(f"{name:15}: {iou*100:6.2f}%")
print("-" * 30)
print(f"MEAN IoU: {miou*100:6.2f}%")
print("="*30)

evaluation on 500 images...
Current model: COCO


100%|██████████| 500/500 [05:16<00:00,  1.58it/s]


Results MIOU - COCO
road           :  94.13%
sidewalk       :  64.84%
building       :  87.72%
wall           :  47.05%
fence          :  49.83%
pole           :   0.00%
traffic light  :  60.64%
traffic sign   :   4.70%
vegetation     :  85.61%
terrain        :   0.44%
sky            :  89.72%
person         :  70.50%
rider          :   0.00%
car            :  84.28%
truck          :  29.35%
bus            :  71.17%
train          :  13.63%
motorcycle     :  63.79%
bicycle        :  73.35%
------------------------------
MEAN IoU:  52.15%


# Evaluation anomaly metrics

In [ ]:
seed = 42

# general reproducibility
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

NUM_CHANNELS = 3
NUM_CLASSES = 20
# gpu training specific
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = True

input_transform = Compose(
    [
        Resize((512, 1024), Image.BILINEAR),
        #ToTensor(),
        # Normalize([.485, .456, .406], [.229, .224, .225]),
    ]
)

target_transform = Compose(
    [
        Resize((512, 1024), Image.NEAREST),
    ]
)


In [ ]:
parser = ArgumentParser()

parser.add_argument(
    "--input",
    #default="/content/drive/MyDrive/CourseProjectAnomaly/Anomaly_Validation_Datasets/Validation_Dataset/RoadAnomaly/images/*.jpg", # change the default path based on the dataset folder
    #default="/content/drive/MyDrive/CourseProjectAnomaly/Anomaly_Validation_Datasets/Validation_Dataset/RoadAnomaly21/images/*.png",
    #default="/content/drive/MyDrive/CourseProjectAnomaly/Anomaly_Validation_Datasets/Validation_Dataset/fs_static/images/*.jpg",
    default="/content/drive/MyDrive/CourseProjectAnomaly/Anomaly_Validation_Datasets/Validation_Dataset/FS_LostFound_full/images/*.png",
    #default="/content/drive/MyDrive/CourseProjectAnomaly/Anomaly_Validation_Datasets/Validation_Dataset/RoadObsticle21/images/*.webp",
    nargs="+",
    help="A list of space separated input images; "
    "or a single glob pattern such as 'directory/*.jpg'",
)

parser.add_argument('--num-workers', type=int, default=2)
parser.add_argument('--batch-size', type=int, default=1)
parser.add_argument('--cpu', action='store_true')
parser.add_argument('--device', type=str, default='cuda', help="cpu or cuda, the device used for evaluation")
args = parser.parse_args(args=[])


anomaly_score_list_maxlogit = []
anomaly_score_list_msp = []
anomaly_score_list_maxentropy = []
anomaly_score_list_rba = []
ood_gts_list = []


In [ ]:
def get_eomt_logits(model, img_tensor, target_size):
    """
    Prende l'immagine, la passa in EoMT usando la logica a finestre
    e restituisce i logits semantici pixel-per-pixel [1, C, H, W].
    """
    model.eval()
    model.window_size = target_size[0]

    with torch.no_grad(), autocast(dtype=torch.float16, device_type="cuda"):
        # La rete si aspetta una lista di tensori
        imgs = [img_tensor.to(device)]
        img_sizes = [img.shape[-2:] for img in imgs]

        # Gestione dei crop (specifico di EoMT) questi metodi fanno parte dell'architettura EOMT
        crops, origins = model.window_imgs_semantic(imgs)

        # Forward pass vero e proprio
        mask_logits_per_layer, class_logits_per_layer = model(crops)

        # Upsampling delle maschere
        mask_logits = F.interpolate(
            mask_logits_per_layer[-1], target_size, mode="bilinear"
        )

        # Fusione di maschere e classi per ottenere i logits dei pixel questi metodi fanno parte dell'architettura EOMT
        crop_logits = model.to_per_pixel_logits_semantic(
            mask_logits, class_logits_per_layer[-1]
        )

        # Ricostruzione dell'immagine intera dai crop questi metodi fanno parte dell'architettura EOMT
        logits = model.revert_window_logits_semantic(crop_logits, origins, img_sizes)

    return logits # Forma: [1, NUM_CLASSI, H, W]

In [ ]:
# Controlliamo se args.input è una lista o una stringa
input_pattern = args.input[0] if isinstance(args.input, list) else args.input

# Creiamo una cartella per salvare i logits (Cruciale per la Temperature Scaling!)
#os.makedirs("saved_logits_coco_RoadAnomaly", exist_ok=True) # da cambiare in base al dataset
# os.makedirs("saved_logits_coco_RoadAnomaly21", exist_ok=True)
# os.makedirs("saved_logits_coco_fs_static", exist_ok=True)
os.makedirs("saved_logits_coco_FS_LostFound_full", exist_ok=True)
#os.makedirs("saved_logits_coco_RoadObsticle21", exist_ok=True)


for path in glob.glob(os.path.expanduser(str(input_pattern))):

    img_pil = input_transform((Image.open(path).convert('RGB')))
    images = torch.from_numpy(np.array(img_pil)).permute(2, 0, 1)

    # Otteniamo i logits dal modello EoMT che restituisce una lista
    logits_list = get_eomt_logits(model, images, target_img_size) # NEW
    logits = logits_list[0].unsqueeze(0) # Estraiamo il tensore dalla lista e gli aggiungiamo la dimensione batch
    # così diventa [1, 19, Altezza, Larghezza]

    # SALVATAGGIO LOGITS SU DISCO (PRO TIP)
    # Li salviamo sulla CPU per non riempire la RAM della scheda video
    nome_file_logit = os.path.basename(path).replace(".jpg", ".pt").replace(".png", ".pt").replace(".webp", ".pt")

    #torch.save(logits.cpu(), os.path.join("saved_logits_coco_RoadAnomaly", nome_file_logit)) # da cambiare in base al dataset
    #torch.save(logits.cpu(), os.path.join("saved_logits_coco_RoadAnomaly21", nome_file_logit))
    #torch.save(logits.cpu(), os.path.join("saved_logits_coco_fs_static", nome_file_logit))
    torch.save(logits.cpu(), os.path.join("saved_logits_coco_FS_LostFound_full", nome_file_logit))
    #torch.save(logits.cpu(), os.path.join("saved_logits_coco_RoadObsticle21", nome_file_logit))


    #compute max logit
    anomaly_result_maxlogit = 1.0 - torch.max(logits.squeeze(0), dim=0)[0].cpu().numpy() # più efficiente

    #compute MSP
    probs = torch.nn.functional.softmax(logits, dim=1)
    max_probs, _ = torch.max(probs.squeeze(0), dim=0)
    anomaly_result_msp = 1.0 - max_probs.data.cpu().numpy()

    #compute Max Entropy
    probs = probs.squeeze(0)
    epsilon = 1e-10
    entropy = -torch.sum(probs * torch.log(probs + epsilon), dim=0)
    anomaly_result_maxentropy = entropy.data.cpu().numpy()

    #compute RbA
    anomaly_result_rba = -torch.sum(torch.tanh(logits.squeeze(0)), dim=0).cpu().numpy() #get_RbA(model, images)

    # gestione maschere
    pathGT = path.replace("images", "labels_masks")
    if "RoadObsticle21" in pathGT:
        pathGT = pathGT.replace("webp", "png")
    if "fs_static" in pathGT:
        pathGT = pathGT.replace("jpg", "png")
    if "RoadAnomaly" in pathGT:
        pathGT = pathGT.replace("jpg", "png")

    mask = Image.open(pathGT)
    mask = target_transform(mask)
    ood_gts = np.array(mask)

    if "RoadAnomaly" in pathGT:
      #in RoadAnomaly 2 indica anomalia --> viene trasformato in 1 per uniformare
        ood_gts = np.where((ood_gts==2), 1, ood_gts)

    if 1 not in np.unique(ood_gts):
        continue   #se non c'è nessuna anomalia, allora salta l'immagine e passa a valutare la successiva
    else:
          ood_gts_list.append(ood_gts)
          anomaly_score_list_maxlogit.append(anomaly_result_maxlogit)
          anomaly_score_list_msp.append(anomaly_result_msp)
          anomaly_score_list_maxentropy.append(anomaly_result_maxentropy)
          anomaly_score_list_rba.append(anomaly_result_rba)

    # per evitare out of memory error
    #del logits, anomaly_result_maxlogit,anomaly_result_msp, anomaly_result_maxentropy, anomaly_result_rba, ood_gts, mask
    del logits_list, logits, probs
    del anomaly_result_maxlogit, anomaly_result_msp, anomaly_result_maxentropy, anomaly_result_rba
    del ood_gts, mask, img_pil, images

    torch.cuda.empty_cache() # Svuota la GPU
    import gc
    gc.collect() # Forza la RAM di sistema a buttare la spazzatura


In [ ]:
ood_gts = np.array(ood_gts_list)
anomaly_scores_maxlogit = np.array(anomaly_score_list_maxlogit)
anomaly_scores_msp = np.array(anomaly_score_list_msp)
anomaly_scores_maxentropy = np.array(anomaly_score_list_maxentropy)
anomaly_scores_rba = np.array(anomaly_score_list_rba)

ood_mask = (ood_gts == 1)
ind_mask = (ood_gts == 0)

# max logit
ood_out = anomaly_scores_maxlogit[ood_mask]
ind_out = anomaly_scores_maxlogit[ind_mask]

ood_label = np.ones(len(ood_out))
ind_label = np.zeros(len(ind_out))

val_out = np.concatenate((ind_out, ood_out))
val_label = np.concatenate((ind_label, ood_label))

prc_auc = average_precision_score(val_label, val_out)
fpr = fpr_at_95_tpr(val_out, val_label)

print(f'AUPRC score maxlogit: {prc_auc*100.0}')
print(f'FPR@TPR95 maxlogit: {fpr*100.0}')


# MSP
ood_out = anomaly_scores_msp[ood_mask]
ind_out = anomaly_scores_msp[ind_mask]

ood_label = np.ones(len(ood_out))
ind_label = np.zeros(len(ind_out))

val_out = np.concatenate((ind_out, ood_out))
val_label = np.concatenate((ind_label, ood_label))

prc_auc = average_precision_score(val_label, val_out)
fpr = fpr_at_95_tpr(val_out, val_label)

print(f'AUPRC score msp: {prc_auc*100.0}')
print(f'FPR@TPR95 msp: {fpr*100.0}')


# max entropy
ood_out = anomaly_scores_maxentropy[ood_mask]
ind_out = anomaly_scores_maxentropy[ind_mask]

ood_label = np.ones(len(ood_out))
ind_label = np.zeros(len(ind_out))

val_out = np.concatenate((ind_out, ood_out))
val_label = np.concatenate((ind_label, ood_label))

prc_auc = average_precision_score(val_label, val_out)
fpr = fpr_at_95_tpr(val_out, val_label)

print(f'AUPRC score maxentropy: {prc_auc*100.0}')
print(f'FPR@TPR95 maxentropy: {fpr*100.0}')


# rba
ood_out = anomaly_scores_rba[ood_mask]
ind_out = anomaly_scores_rba[ind_mask]

ood_label = np.ones(len(ood_out))
ind_label = np.zeros(len(ind_out))

val_out = np.concatenate((ind_out, ood_out))
val_label = np.concatenate((ind_label, ood_label))

prc_auc = average_precision_score(val_label, val_out)
fpr = fpr_at_95_tpr(val_out, val_label)

print(f'AUPRC score rba: {prc_auc*100.0}')
print(f'FPR@TPR95 rba: {fpr*100.0}')


AUPRC score maxlogit: 3.330692649023428
FPR@TPR95 maxlogit: 97.27686058046194
AUPRC score msp: 3.33104175085702
FPR@TPR95 msp: 97.24967008252223
AUPRC score maxentropy: 3.5011159468269373
FPR@TPR95 maxentropy: 97.1525678780841
AUPRC score rba: 0.21662296293572
FPR@TPR95 rba: 97.02816861474743


# Temperature scaling


In [ ]:
#cartella_logits = "saved_logits_coco_RoadAnomaly" # cambiare in base al dataset
# cartella_logits = "saved_logits_coco_RoadAnomaly21"
# cartella_logits = "saved_logits_coco_fs_static"
cartella_logits = "saved_logits_coco_FS_LostFound_full"
#cartella_logits = "saved_logits_coco_RoadObsticle21"


#cartella_maschere = "/content/drive/MyDrive/CourseProjectAnomaly/Anomaly_Validation_Datasets/Validation_Dataset/RoadAnomaly/labels_masks" # da cambiare in base al dataset
#cartella_maschere = "/content/drive/MyDrive/CourseProjectAnomaly/Anomaly_Validation_Datasets/Validation_Dataset/RoadAnomaly21/labels_masks"
#cartella_maschere = "/content/drive/MyDrive/CourseProjectAnomaly/Anomaly_Validation_Datasets/Validation_Dataset/fs_static/labels_masks"
cartella_maschere = "/content/drive/MyDrive/CourseProjectAnomaly/Anomaly_Validation_Datasets/Validation_Dataset/FS_LostFound_full/labels_masks"
#cartella_maschere = "/content/drive/MyDrive/CourseProjectAnomaly/Anomaly_Validation_Datasets/Validation_Dataset/RoadObsticle21/labels_masks"

# Valori di Temperatura da esplorare
# T=1.0 è la baseline (dovrebbe darti lo stesso identico valore che hai ottenuto nel ciclo)
temperature_da_testare = [0.01, 0.05, 0.1, 0.2, 0.5, 1.0, 1.5, 2.0, 5.0, 10.0, 100.0]

for T in temperature_da_testare:
    score_msp_T = []
    gts_T = []

    # Cerchiamo tutti i file .pt salvati
    for logit_path in glob.glob(os.path.join(cartella_logits, "*.pt")):

        # 1. Carica i logits grezzi [1, C, H, W]
        logits = torch.load(logit_path)

        # 2. APPLICA LA TEMPERATURA!
        # Dividiamo i logits per T prima di applicare la softmax
        probs = torch.nn.functional.softmax(logits / T, dim=1)

        # 3. Calcola l'Anomaly Score (MSP)
        max_probs, _ = torch.max(probs.squeeze(0), dim=0)
        anomaly_msp = 1.0 - max_probs.numpy()

        # 4. Recupera la maschera (Ground Truth)
        nome_base = os.path.basename(logit_path).replace(".pt", "")
        pathGT = os.path.join(cartella_maschere, f"{nome_base}.png")

        if not os.path.exists(pathGT):
            continue

        mask = Image.open(pathGT)
        mask = target_transform(mask)
        ood_gts = np.array(mask)

        if "RoadAnomaly" in cartella_maschere:
            ood_gts = np.where((ood_gts==2), 1, ood_gts)

        if 1 in np.unique(ood_gts):
            gts_T.append(ood_gts)
            score_msp_T.append(anomaly_msp)

    # 5. Calcolo AUPRC e FPR95 per questa Temperatura
    if len(gts_T) > 0:
        val_gts = np.concatenate(gts_T)
        val_msp = np.concatenate(score_msp_T)

        ood_mask = (val_gts == 1)
        ind_mask = (val_gts == 0)

        val_out_msp = np.concatenate((val_msp[ind_mask], val_msp[ood_mask]))
        val_label = np.concatenate((np.zeros(np.sum(ind_mask)), np.ones(np.sum(ood_mask))))

        auprc_msp = average_precision_score(val_label, val_out_msp)
        fpr_msp = fpr_at_95_tpr(val_out_msp, val_label)

        print(f"=== Temperature T = {T} ===")
        print(f"AUPRC MSP: {auprc_msp*100:.2f}%  |  FPR95 MSP: {fpr_msp*100:.2f}%\n")
    else:
        print(f"Warning: No anomaly mask for T={T}")

=== Temperature T = 0.01 ===
AUPRC MSP: 0.35%  |  FPR95 MSP: 94.24%

=== Temperature T = 0.05 ===
AUPRC MSP: 0.78%  |  FPR95 MSP: 95.61%

=== Temperature T = 0.1 ===
AUPRC MSP: 2.06%  |  FPR95 MSP: 97.45%

=== Temperature T = 0.2 ===
AUPRC MSP: 3.20%  |  FPR95 MSP: 97.33%

=== Temperature T = 0.5 ===
AUPRC MSP: 3.32%  |  FPR95 MSP: 97.20%

=== Temperature T = 1.0 ===
AUPRC MSP: 3.33%  |  FPR95 MSP: 97.25%

=== Temperature T = 1.5 ===
AUPRC MSP: 3.33%  |  FPR95 MSP: 97.25%

=== Temperature T = 2.0 ===
AUPRC MSP: 3.33%  |  FPR95 MSP: 97.25%

=== Temperature T = 5.0 ===
AUPRC MSP: 3.34%  |  FPR95 MSP: 97.25%

=== Temperature T = 10.0 ===
AUPRC MSP: 3.34%  |  FPR95 MSP: 97.25%

=== Temperature T = 100.0 ===
AUPRC MSP: 3.35%  |  FPR95 MSP: 97.25%

